<a href="https://colab.research.google.com/github/ladkrutarth/Hands_on_Stable_Diffusion/blob/main/Hands_on_Stable_Diffusion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Hairstyle & Makeup Style Generator
### CS 5588 · Data Science Capstone · Option 1

**Pipeline:** structured `BeautyAttributes` input → prompt engineering → SDXL generation → CLIP evaluation

| Cell | Purpose |
|------|---------|
| 1 | Install dependencies |
| 2 | Imports + configuration |
| 3 | Data models & style catalogs |
| 4 | Prompt engineering functions |
| 5 | Generation & visualization utilities |
| 6 | Load SDXL pipeline |
| 7 | CLIP evaluator |
| 8 | Run pipeline (generate + display) |
| 9 | Evaluate with CLIP + plot results |

## Cell 1 · Install Dependencies

In [ ]:
%pip install -q torch torchvision diffusers transformers accelerate safetensors pillow matplotlib open_clip_torch

## Cell 2 · Imports & Configuration
All imports and tuneable constants live here — change once, effective everywhere.

In [ ]:
# ── Standard library ─────────────────────────────────────────────────────────
import math
import os
import random
from dataclasses import dataclass, field
from typing import List, Optional, Tuple

# ── Third-party ───────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import numpy as np
import open_clip
import torch
from diffusers import DiffusionPipeline          # SDXL uses DiffusionPipeline, NOT StableDiffusionPipeline
from PIL import Image

# ── Device detection ─────────────────────────────────────────────────────────
if "COLAB_TPU_ADDR" in os.environ:
    try:
        import torch_xla.core.xla_model as xm
        DEVICE: str = str(xm.xla_device())
    except ImportError:
        print("TPU detected but torch_xla not installed — falling back to CPU.")
        DEVICE = "cpu"
elif torch.cuda.is_available():
    DEVICE = "cuda"
elif getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

# ── Model IDs ─────────────────────────────────────────────────────────────────
SDXL_MODEL_ID  = "stabilityai/stable-diffusion-xl-base-1.0"
CLIP_MODEL_ID  = "ViT-B-32"
CLIP_PRETRAIN  = "openai"

# ── Generation defaults (change here, applied everywhere) ────────────────────
@dataclass
class GenConfig:
    num_images:          int   = 4      # images per prompt (set to 2 on CPU for speed)
    num_inference_steps: int   = 35
    guidance_scale:      float = 7.5
    height:              int   = 1024
    width:               int   = 1024
    base_seed:           int   = 42

    def __post_init__(self) -> None:
        if DEVICE == "cpu":
            # Reduce load for CPU-only environments
            object.__setattr__(self, "num_images", 2)
            object.__setattr__(self, "num_inference_steps", 20)
            object.__setattr__(self, "height", 512)
            object.__setattr__(self, "width", 512)
            print("[GenConfig] CPU detected — reduced to 2 images @ 512px, 20 steps.")

GEN_CFG = GenConfig()

print(f"Device : {DEVICE}")
print(f"SDXL   : {SDXL_MODEL_ID}")
print(f"Config : {GEN_CFG}")

## Cell 3 · Data Models & Style Catalogs
`BeautyAttributes` is the single source of truth for all style parameters.

In [ ]:
# ── Negative prompt (shared by all generation calls) ─────────────────────────
NEGATIVE_PROMPT: str = (
    "lowres, blurry, worst quality, jpeg artifacts, "
    "duplicate faces, distorted eyes, asymmetric eyes, "
    "deformed hands, extra fingers, watermark, text"
)

# ── Input schema ──────────────────────────────────────────────────────────────
@dataclass
class BeautyAttributes:
    """Structured representation of a styling request.

    Parameters
    ----------
    gender       : "man" | "woman"
    hairstyle    : e.g. "sleek low bun", "short textured fade"
    hair_color   : e.g. "dark brown", "platinum blonde"
    makeup_style : e.g. "soft glam with rose lipstick", "clean grooming"
    occasion     : any entry from OCCASIONS below
    """
    gender:       str
    hairstyle:    str
    hair_color:   str
    makeup_style: str
    occasion:     str

    _VALID_GENDERS = frozenset({"man", "woman"})

    def __post_init__(self) -> None:
        if self.gender not in self._VALID_GENDERS:
            raise ValueError(f"gender must be one of {self._VALID_GENDERS!r}, got {self.gender!r}")
        # Normalise whitespace
        for attr in ("hairstyle", "hair_color", "makeup_style", "occasion"):
            object.__setattr__(self, attr, " ".join(getattr(self, attr).split()))

    def __str__(self) -> str:
        return (
            f"{self.gender} | {self.hairstyle} | "
            f"{self.hair_color} | {self.makeup_style} | {self.occasion}"
        )


# ── Style catalogs ────────────────────────────────────────────────────────────
OCCASIONS: List[str] = [
    # Professional
    "job interviews", "office formal day", "business meetings",
    "conferences and seminars", "networking events", "presentations",
    "casual office day", "team outings", "client lunches",
    "workshops and training sessions",
    # Social
    "birthday parties", "house parties", "dinner with friends",
    "dates", "clubbing and nightlife",
    # Formal / ceremony
    "weddings", "engagement ceremonies", "anniversaries",
    "award functions", "gala events",
    # Cultural / religious
    "festivals such as Diwali or Christmas",
    "religious ceremonies", "family functions", "cultural events",
    # Leisure / travel
    "vacations", "beach outings", "road trips", "sightseeing",
    # Athletic
    "gym workout", "running and jogging", "sports matches", "yoga sessions",
    # Everyday
    "working from home", "relaxing at home",
    "running errands", "grocery shopping",
    # Solemn
    "funerals", "memorial services",
    # Academic
    "school or college classes", "graduation ceremonies",
    "convocations",
    # Special
    "first date", "photoshoots", "public speaking",
]

WOMAN_HAIRSTYLES: List[str] = [
    "loose beach waves", "sleek ponytail", "short bob",
    "elegant updo", "sleek low bun", "braided crown",
    "half-up half-down", "voluminous blowout",
]

MAN_HAIRSTYLES: List[str] = [
    "short textured fade", "neat side part", "clean fade",
    "slicked-back undercut", "modern quiff", "buzz cut",
]

HAIR_COLORS: List[str] = [
    "black", "dark brown", "chestnut brown", "auburn",
    "dark blonde", "platinum blonde", "silver gray",
]

WOMAN_MAKEUP: List[str] = [
    "soft glam with rose lipstick", "smokey eye with matte red lip",
    "natural no-makeup look", "bold cat-eye liner",
    "dewy skin with coral blush",
]

MAN_GROOMING: List[str] = [
    "clean grooming", "light stubble well-groomed",
    "freshly shaved clean-cut", "neatly trimmed beard",
]

CLOTHING_COLORS: List[str] = [
    "black", "charcoal gray", "navy blue",
    "burgundy", "emerald green", "cream", "deep plum",
]


def random_attributes(gender: Optional[str] = None) -> BeautyAttributes:
    """Return a randomly sampled BeautyAttributes instance."""
    g = gender or random.choice(["man", "woman"])
    return BeautyAttributes(
        gender=g,
        hairstyle=random.choice(WOMAN_HAIRSTYLES if g == "woman" else MAN_HAIRSTYLES),
        hair_color=random.choice(HAIR_COLORS),
        makeup_style=random.choice(WOMAN_MAKEUP if g == "woman" else MAN_GROOMING),
        occasion=random.choice(OCCASIONS),
    )


# ── Smoke-test ────────────────────────────────────────────────────────────────
for g in ["woman", "man"]:
    print(f"Sample {g}: {random_attributes(g)}")

## Cell 4 · Prompt Engineering

Three prompt builders — all operate on `BeautyAttributes`, return `str`.

| Function | Description |
|----------|-------------|
| `build_naive_prompt` | Bag-of-words baseline |
| `build_structured_prompt` | Basic structured with photography context |
| `build_professional_prompt` | Full editorial-grade prompt with clothing color |

In [ ]:
def build_naive_prompt(attrs: BeautyAttributes) -> str:
    """Bag-of-words baseline — mimics a typical unstructured user query."""
    return (
        f"{attrs.gender} {attrs.hairstyle} {attrs.hair_color} hair "
        f"{attrs.makeup_style} {attrs.occasion} face photo"
    )


def build_structured_prompt(attrs: BeautyAttributes) -> str:
    """Structured prompt: complete grammar + photography context + quality tags."""
    return (
        f"professional portrait photograph of one adult {attrs.gender}, "
        f"{attrs.hairstyle} hairstyle, {attrs.hair_color} hair, "
        f"{attrs.makeup_style}, suitable for {attrs.occasion}, "
        f"natural skin texture, soft diffused lighting, 85mm lens, "
        f"shallow depth of field, high detail, realistic, "
        f"editorial beauty photography"
    )


def build_professional_prompt(
    attrs: BeautyAttributes,
    clothing_color: str = "black",
) -> str:
    """Full editorial prompt — adds clothing color and cinematic grading."""
    return (
        f"high-end editorial portrait of a {attrs.gender}, "
        f"{attrs.hairstyle} {attrs.hair_color} hair, "
        f"professional {attrs.makeup_style}, "
        f"wearing elegant {clothing_color} formal attire for {attrs.occasion}, "
        f"sharp focus, hyper-realistic skin textures, 85mm lens f/1.8, "
        f"soft studio lighting, 8k UHD, cinematic color grading, "
        f"sophisticated atmosphere"
    )


# ── Demo ──────────────────────────────────────────────────────────────────────
def _demo_prompts() -> None:
    examples = [
        BeautyAttributes("woman", "sleek low bun", "dark brown",
                         "soft glam with rose lipstick", "evening gala"),
        BeautyAttributes("man",   "clean side part", "black",
                         "clean grooming", "business meetings"),
    ]
    for a in examples:
        print(f"=== {a.gender.upper()} ===")
        print(f"  NAIVE      : {build_naive_prompt(a)}")
        print(f"  STRUCTURED : {build_structured_prompt(a)}")
        print(f"  PROFESSIONAL: {build_professional_prompt(a, 'navy blue')}")
        print()

_demo_prompts()

## Cell 5 · Generation & Visualization Utilities

`generate_images` now receives **`pipe`** as an explicit parameter — no hidden global dependency.

In [ ]:
@torch.inference_mode()
def generate_images(
    pipe:          DiffusionPipeline,
    prompt:        str,
    cfg:           GenConfig          = GEN_CFG,
    negative:      str                = NEGATIVE_PROMPT,
) -> List[Image.Image]:
    """Generate `cfg.num_images` images from *prompt*, one per sequential seed.

    Parameters
    ----------
    pipe     : loaded DiffusionPipeline (SDXL or compatible)
    prompt   : text prompt
    cfg      : generation hyper-parameters (default: module-level GEN_CFG)
    negative : negative prompt string

    Returns
    -------
    List of PIL Images, length == cfg.num_images
    """
    images: List[Image.Image] = []
    generator = torch.Generator(device=DEVICE)

    for i in range(cfg.num_images):
        generator.manual_seed(cfg.base_seed + i)
        out = pipe(
            prompt=prompt,
            negative_prompt=negative,
            num_inference_steps=cfg.num_inference_steps,
            guidance_scale=cfg.guidance_scale,
            height=cfg.height,
            width=cfg.width,
            generator=generator,
        )
        images.append(out.images[0])

    return images


def show_grid(
    images:       List[Image.Image],
    titles:       Optional[List[str]] = None,
    figsize_per:  float               = 3.2,
    suptitle:     Optional[str]       = None,
) -> None:
    """Display *images* in a grid with optional per-image *titles*."""
    n    = len(images)
    cols = min(4, n)
    rows = math.ceil(n / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(cols * figsize_per, rows * figsize_per))

    # Normalise axes to a flat list regardless of shape
    if n == 1:
        axes_flat: List = [axes]
    elif rows == 1:
        axes_flat = list(axes)
    else:
        axes_flat = list(axes.flatten())

    for ax, img in zip(axes_flat, images):
        ax.imshow(img)
        ax.axis("off")
    for ax in axes_flat[n:]:
        ax.axis("off")

    if titles:
        for ax, title in zip(axes_flat[:n], titles):
            truncated = (title[:78] + "…") if len(title) > 80 else title
            ax.set_title(truncated, fontsize=7)

    if suptitle:
        fig.suptitle(suptitle, fontsize=10, fontweight="bold", y=1.01)

    plt.tight_layout()
    plt.show()


def show_comparison_grid(
    imgs_a:    List[Image.Image],
    imgs_b:    List[Image.Image],
    label_a:   str = "Structured",
    label_b:   str = "Naive",
    cfg:       GenConfig = GEN_CFG,
) -> None:
    """Interleave structured and naive images: A0 B0 A1 B1 ... for side-by-side comparison."""
    paired: List[Image.Image] = []
    titles: List[str]         = []
    for i, (a, b) in enumerate(zip(imgs_a, imgs_b)):
        paired.extend([a, b])
        titles.extend([f"{label_a} #{i} (seed {cfg.base_seed + i})",
                        f"{label_b} #{i} (seed {cfg.base_seed + i})"])
    show_grid(paired, titles=titles, suptitle=f"Left = {label_a}   |   Right = {label_b}")


print("Utilities loaded: generate_images, show_grid, show_comparison_grid")

## Cell 6 · Load SDXL Pipeline

`load_sdxl()` is a **pure function** — returns the pipeline, stores nothing globally except `sdxl_pipe`.

In [ ]:
def load_sdxl(model_id: str = SDXL_MODEL_ID, device: str = DEVICE) -> DiffusionPipeline:
    """Load and configure SDXL base pipeline.

    Memory strategy
    ---------------
    CUDA  → float16 + enable_model_cpu_offload()  (saves ~4 GB VRAM)
    MPS   → float32, no offload (MPS doesn't support offload)
    CPU   → float32, attention slicing for speed

    Note: enable_model_cpu_offload() moves the pipe to CUDA internally;
    do NOT call .to(device) afterwards — that call is handled by the offloader.
    """
    dtype = torch.float16 if device == "cuda" else torch.float32
    print(f"Loading {model_id}  |  device={device}  |  dtype={dtype}")

    pipe = DiffusionPipeline.from_pretrained(
        model_id,
        torch_dtype=dtype,
        use_safetensors=True,
    )

    if device == "cuda":
        # cpu_offload streams sub-models to GPU on demand — do NOT call .to("cuda") after this
        pipe.enable_model_cpu_offload()
    elif device == "mps":
        pipe = pipe.to(device)
    else:  # CPU
        pipe = pipe.to(device)
        if hasattr(pipe, "enable_attention_slicing"):
            pipe.enable_attention_slicing()

    print("SDXL pipeline ready.")
    return pipe


sdxl_pipe = load_sdxl()

## Cell 7 · CLIP Evaluator

`CLIPEvaluator` encapsulates model + tokenizer — no module-level globals.

In [ ]:
class CLIPEvaluator:
    """Compute CLIP cosine-similarity between images and a text prompt.

    Usage
    -----
    evaluator = CLIPEvaluator()
    scores = evaluator.score(images, prompt)   # → List[float]
    stats  = evaluator.stats(images, prompt)   # → dict with mean, std, scores
    """

    def __init__(
        self,
        model_name: str = CLIP_MODEL_ID,
        pretrained: str = CLIP_PRETRAIN,
        device:     str = DEVICE,
    ) -> None:
        self.device = device
        print(f"Loading CLIP {model_name} (pretrained={pretrained}) on {device} ...")
        self.model, _, self.preprocess = open_clip.create_model_and_transforms(
            model_name, pretrained=pretrained, device=device
        )
        self.tokenizer = open_clip.get_tokenizer(model_name)
        self.model.eval()
        print("CLIP evaluator ready.")

    @torch.inference_mode()
    def score(
        self,
        images: List[Image.Image],
        text:   str,
    ) -> List[float]:
        """Return per-image cosine similarity to *text*."""
        tokens    = self.tokenizer([text]).to(self.device)
        text_feat = self.model.encode_text(tokens)
        text_feat = text_feat / text_feat.norm(dim=-1, keepdim=True)

        scores: List[float] = []
        for img in images:
            tensor   = self.preprocess(img).unsqueeze(0).to(self.device)
            img_feat = self.model.encode_image(tensor)
            img_feat = img_feat / img_feat.norm(dim=-1, keepdim=True)
            scores.append(float((img_feat @ text_feat.T).item()))

        return scores

    def stats(
        self,
        images: List[Image.Image],
        text:   str,
    ) -> dict:
        """Return a dict with keys: scores, mean, std."""
        s = self.score(images, text)
        return {"scores": s, "mean": float(np.mean(s)), "std": float(np.std(s))}


clip_eval = CLIPEvaluator()

## Cell 8 · Run Pipeline
Pick style attributes → build both prompts → generate → display comparison grid.

In [ ]:
# ── 1. Sample attributes ──────────────────────────────────────────────────────
# You can also set these explicitly, e.g.:
#   attrs = BeautyAttributes("woman", "sleek low bun", "dark brown",
#                            "soft glam with rose lipstick", "evening gala")
attrs          = random_attributes()               # random pick from catalogs
clothing_color = random.choice(CLOTHING_COLORS)

print(f"Selected attributes : {attrs}")
print(f"Clothing color      : {clothing_color}")

# ── 2. Build prompts ──────────────────────────────────────────────────────────
p_structured   = build_structured_prompt(attrs)
p_naive        = build_naive_prompt(attrs)
p_professional = build_professional_prompt(attrs, clothing_color)

print(f"\nStructured   : {p_structured}")
print(f"Naive        : {p_naive}")
print(f"Professional : {p_professional}")

# ── 3. Generate images ────────────────────────────────────────────────────────
print(f"\nGenerating structured prompt images  (seed {GEN_CFG.base_seed}–{GEN_CFG.base_seed + GEN_CFG.num_images - 1}) ...")
imgs_structured   = generate_images(sdxl_pipe, p_structured,   cfg=GEN_CFG)

print("Generating naive prompt images ...")
imgs_naive        = generate_images(sdxl_pipe, p_naive,         cfg=GEN_CFG)

print("Generating professional prompt images ...")
imgs_professional = generate_images(sdxl_pipe, p_professional,  cfg=GEN_CFG)

# ── 4. Display comparison grid ────────────────────────────────────────────────
print("\n--- Structured vs Naive ---")
show_comparison_grid(imgs_structured, imgs_naive, "Structured", "Naive", cfg=GEN_CFG)

print("\n--- Professional ---")
show_grid(
    imgs_professional,
    titles=[f"professional #{i}" for i in range(len(imgs_professional))],
    suptitle=f"Professional prompt  |  {attrs.gender}  |  {attrs.occasion}  |  {clothing_color}",
)

## Cell 9 · Evaluate with CLIP + Plot Results

Scores all three prompt types, prints a stats table, and draws a bar chart for easy comparison.

In [ ]:
# ── Compute CLIP stats ────────────────────────────────────────────────────────
results = {
    "Structured":    clip_eval.stats(imgs_structured,   p_structured),
    "Naive":         clip_eval.stats(imgs_naive,         p_naive),
    "Professional":  clip_eval.stats(imgs_professional,  p_professional),
}

# ── Print summary table ───────────────────────────────────────────────────────
print(f"{'Prompt Type':<14}  {'Scores':<42}  {'Mean':>6}  {'Std':>6}")
print("-" * 74)
for label, r in results.items():
    score_str = str([round(s, 4) for s in r["scores"]])
    print(f"{label:<14}  {score_str:<42}  {r['mean']:>6.4f}  {r['std']:>6.4f}")

# ── Bar chart: per-image CLIP scores ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: individual scores per seed
n     = GEN_CFG.num_images
seeds = [f"seed {GEN_CFG.base_seed + i}" for i in range(n)]
x     = np.arange(n)
width = 0.28
colors = ["#0ea5e9", "#f72585", "#06d6a0"]

for idx, (label, r) in enumerate(results.items()):
    axes[0].bar(x + idx * width, r["scores"], width, label=label, color=colors[idx], alpha=0.85)

axes[0].set_xticks(x + width)
axes[0].set_xticklabels(seeds, fontsize=8)
axes[0].set_ylabel("CLIP Cosine Similarity")
axes[0].set_title("Per-image CLIP Scores by Prompt Type")
axes[0].legend(fontsize=8)
axes[0].set_ylim(0.22, 0.32)
axes[0].grid(axis="y", linestyle="--", alpha=0.4)

# Right: mean ± std per prompt type
labels = list(results.keys())
means  = [results[l]["mean"] for l in labels]
stds   = [results[l]["std"]  for l in labels]

bars = axes[1].bar(labels, means, color=colors, alpha=0.85, capsize=5)
axes[1].errorbar(labels, means, yerr=stds, fmt="none", color="#111", capsize=6, linewidth=1.5)
axes[1].set_ylabel("Mean CLIP Score")
axes[1].set_title("Mean ± Std CLIP Score by Prompt Type")
axes[1].set_ylim(0.22, 0.32)
axes[1].grid(axis="y", linestyle="--", alpha=0.4)

# Annotate bars with values
for bar, mean, std in zip(bars, means, stds):
    axes[1].text(
        bar.get_x() + bar.get_width() / 2,
        mean + std + 0.001,
        f"{mean:.4f}",
        ha="center", va="bottom", fontsize=9, fontweight="bold"
    )

fig.suptitle(
    f"CLIP Alignment Analysis  |  {attrs.gender}  |  {attrs.occasion}",
    fontsize=11, fontweight="bold"
)
plt.tight_layout()
plt.show()

# ── Interpretation note ───────────────────────────────────────────────────────
best_label = max(results, key=lambda l: results[l]["mean"])
most_consistent = min(results, key=lambda l: results[l]["std"])
print(f"\nHighest mean CLIP score   : {best_label} ({results[best_label]['mean']:.4f})")
print(f"Most consistent (low std) : {most_consistent} (std={results[most_consistent]['std']:.4f})")
print(
    "\nNote: CLIP cosine similarity favours token-dense short prompts, so naive\n"
    "prompts can score higher despite lower visual richness. Consistency (low\n"
    "std across seeds) is a more reliable proxy for production quality."
)